In [ ]:
pip install findspark

In [6]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1433

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


Successfully connected to HYPE on port 1433.


In [7]:
import os
import findspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") # Added confirmation message




Findspark initialized successfully.


In [15]:
import os
import findspark
from pyspark.sql import SparkSession
import pyspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") 

# Define the paths to your local JAR files, now with the file:/// scheme
# Use forward slashes exclusively in the final path string
mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-13.2.1.jre11.jar"
mysql_jar_path = r"file:///C:/spark_drivers/mysql-connector-j-9.5.0.jar"

spark = SparkSession.builder \
    .appName("SQLServerToMySQLTransfer") \
    .config("spark.jars", f"{mssql_jar_path},{mysql_jar_path}") \
    .getOrCreate()

print("Spark session created.")

# You can now proceed with your data transfer logic here
# For example:
# df = spark.read.format("jdbc").option(...)


Findspark initialized successfully.
Spark session created.


In [12]:
# Connection details for SQL Server (Source)
# Add 'encrypt=true;trustServerCertificate=true' to the URL properties
sql_server_url = r"jdbc:sqlserver://HYPE\SQL_2022:1433;databaseName=dwh;encrypt=true;trustServerCertificate=true"
sql_server_user = "sa"
sql_server_password = "D@1ry$2014"
sql_server_table = "dim_distributor"

# Read data into a PySpark DataFrame
df = spark.read.format("jdbc").options(
    url=sql_server_url,
    dbtable=sql_server_table,
    user=sql_server_user,
    password=sql_server_password,
    driver="com.microsoft.sqlserver.jdbc.SQLServerDriver"
).load()

# Optional: Display the DataFrame to check if data is loaded correctly
df.show()
print(f"Number of rows read from SQL Server: {df.count()}")

# ... (rest of your MySQL writing code) ...


+---------+------------------+--------------------+---------+-------------------+-----------------+------------+-----------+----------------+-----------------+--------------+
| DistCode|          DistName|             FileLoc|     FlNm|prodNameColumnIndex|custIdColumnIndex|maxBlankRows|  DtcColHdr|InvNoColumnIndex|TrDateColumnIndex|QtyColumnIndex|
+---------+------------------+--------------------+---------+-------------------+-----------------+------------+-----------+----------------+-----------------+--------------+
|150001191|Karya_Sukses_Prima|c:\Etl_Danone\Kar...|sales.xls|                  2|               10|          29|       NMBR|               1|                0|             3|
|150030255|     Fajar_Lestari|c:\Etl_Danone\Faj...|sales.xls|                  1|                2|          29|KODE PRODUK|               3|                0|             6|
+---------+------------------+--------------------+---------+-------------------+-----------------+------------+-----------+-

In [13]:
# Connection details for MySQL (Destination)
mysql_url = "jdbc:mysql://localhost:3306/AdventureSales"
mysql_user = "root"
mysql_password = ""
mysql_table = "dim_distributor"

# Write the DataFrame to the MySQL table
df.write.format("jdbc").options(
    url=mysql_url,
    dbtable=mysql_table,
    user=mysql_user,
    password=mysql_password,
    driver="com.mysql.cj.jdbc.Driver"
).mode("overwrite").save()

print("Data written to MySQL successfully.")

df_verify = spark.read.format("jdbc").options(
    url=mysql_url,
    dbtable=mysql_table,
    user=mysql_user,
    password=mysql_password,
    driver="com.mysql.cj.jdbc.Driver"
).load()

# Count and display the number of records in the destination table
mysql_count = df_verify.count()
print(f"Number of rows read from SQL Server (Source): {df.count()}")
print(f"Number of rows written to MySQL (Destination): {mysql_count}")

# Optional: Display top 5 rows of the destination table to confirm content
df_verify.show(5) 

# Stop the Spark session (optional but good practice)
spark.stop()
#print(f"Number of rows read from My sql: {df.count()}")

Data written to MySQL successfully.
Number of rows read from SQL Server (Source): 2
Number of rows written to MySQL (Destination): 2
+---------+------------------+--------------------+---------+-------------------+-----------------+------------+-----------+----------------+-----------------+--------------+
| DistCode|          DistName|             FileLoc|     FlNm|prodNameColumnIndex|custIdColumnIndex|maxBlankRows|  DtcColHdr|InvNoColumnIndex|TrDateColumnIndex|QtyColumnIndex|
+---------+------------------+--------------------+---------+-------------------+-----------------+------------+-----------+----------------+-----------------+--------------+
|150001191|Karya_Sukses_Prima|c:\Etl_Danone\Kar...|sales.xls|                  2|               10|          29|       NMBR|               1|                0|             3|
|150030255|     Fajar_Lestari|c:\Etl_Danone\Faj...|sales.xls|                  1|                2|          29|KODE PRODUK|               3|                0|        